# Delay Prediction Model Endpoint Test

**Model:** Rule-based heuristic delay predictor  
**Endpoint:** `GET /api/predictions/delays`  
**Purpose:** Predict arrival/departure delays based on flight state, time-of-day, and congestion

## Model Overview

The delay prediction model uses rule-based heuristics to estimate flight delays:

| Factor | Condition | Delay Impact | Confidence Impact |
|--------|-----------|--------------|-------------------|
| Peak morning | hour in [7, 8, 9] | +15 min | -0.1 |
| Peak evening | hour in [17, 18, 19] | +12 min | -0.1 |
| Weekend | day_of_week >= 5 | -3 min | +0.05 |
| Ground | altitude = "ground" | +8 min | +0.1 |
| Low altitude | altitude = "low" | +3 min | 0 |
| Cruising | altitude = "cruise" | -2 min | -0.1 |
| Slow moving | velocity < 0.1 (norm) | +5 min | 0 |

## Delay Categories

| Category | Delay Range |
|----------|-------------|
| `on_time` | < 5 minutes |
| `slight` | 5-15 minutes |
| `moderate` | 15-30 minutes |
| `severe` | > 30 minutes |

## Feature Engineering

14 features extracted from each flight:
- **Numeric (4):** hour_of_day (0-1), day_of_week (0-1), is_weekend (0/1), velocity_normalized (0-1)
- **One-hot distance (3):** short, medium, long
- **One-hot altitude (3):** ground, low, cruise
- **One-hot heading (4):** N, E, S, W quadrants

In [ ]:
# Setup & Authentication
import json
import time

import httpx
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# --- Configuration ---
APP_BASE_URL = "https://airport-digital-twin-dev-7474645572615955.aws.databricksapps.com"

# --- Authentication ---
def get_token():
    try:
        from databricks.sdk import WorkspaceClient
        w = WorkspaceClient()
        return w.config.authenticate()["Authorization"].removeprefix("Bearer ")
    except Exception:
        pass
    try:
        import subprocess
        result = subprocess.run(
            ["databricks", "auth", "token", "--profile", "FEVM_SERVERLESS_STABLE"],
            capture_output=True, text=True, timeout=15,
        )
        if result.returncode == 0:
            return json.loads(result.stdout)["access_token"]
    except Exception:
        pass
    raise RuntimeError("No Databricks auth token available")

TOKEN = get_token()
HEADERS = {"Authorization": f"Bearer {TOKEN}"}
print(f"Authenticated (token: {len(TOKEN)} chars)")
print(f"App: {APP_BASE_URL}")

## Endpoint Parameters

### `GET /api/predictions/delays`

**Query parameters:**

| Parameter | Type | Required | Description |
|-----------|------|----------|-------------|
| `icao24` | string | No | ICAO24 address for single flight. If omitted, returns predictions for all active flights |

**Response schema: `DelaysListResponse`**

```json
{
  "delays": [
    {
      "icao24": "a12345",
      "delay_minutes": 15.5,
      "confidence": 0.85,
      "category": "slight"
    }
  ],
  "count": 50
}
```

| Field | Type | Description |
|-------|------|-------------|
| `icao24` | string | Flight ICAO24 identifier |
| `delay_minutes` | float | Predicted delay in minutes (0+) |
| `confidence` | float | Confidence score (0.0 - 1.0) |
| `category` | string | `on_time`, `slight`, `moderate`, or `severe` |

In [ ]:
# Call Delay Prediction Endpoint

url = f"{APP_BASE_URL}/api/predictions/delays"
# Optional: filter to a single flight
# url = f"{APP_BASE_URL}/api/predictions/delays?icao24=a12345"

print(f"GET {url}")

t0 = time.monotonic()
resp = httpx.get(url, headers=HEADERS, timeout=30)
elapsed_ms = int((time.monotonic() - t0) * 1000)

print(f"Status: {resp.status_code} ({elapsed_ms}ms)")

if resp.status_code != 200:
    print(f"Error: {resp.text[:500]}")
    delays = []
else:
    data = resp.json()
    delays = data.get("delays", [])
    print(f"Flights with predictions: {data.get('count', len(delays))}")

    # Show first 10 as a table
    header = "| ICAO24 | Delay (min) | Confidence | Category |"
    sep = "|--------|-------------|------------|----------|"
    rows = [header, sep]
    for d in delays[:15]:
        rows.append(
            f"| `{d['icao24']}` | {d['delay_minutes']:.1f} | {d['confidence']:.2f} | {d['category']} |"
        )
    if len(delays) > 15:
        rows.append(f"| ... | *{len(delays) - 15} more* | | |")
    display(HTML("<pre>" + "\n".join(rows) + "</pre>"))

In [ ]:
# Display Results: Delay Distribution & Category Breakdown

if not delays:
    print("No delay predictions available. Is the app running with active flights?")
else:
    delay_values = [d["delay_minutes"] for d in delays]
    categories = [d["category"] for d in delays]
    confidences = [d["confidence"] for d in delays]

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # --- Histogram of delay minutes ---
    axes[0].hist(delay_values, bins=20, color="steelblue", edgecolor="white", alpha=0.8)
    axes[0].set_xlabel("Delay (minutes)")
    axes[0].set_ylabel("Flight count")
    axes[0].set_title("Delay Distribution")
    axes[0].axvline(x=5, color="green", linestyle="--", alpha=0.5, label="On-time threshold")
    axes[0].axvline(x=15, color="orange", linestyle="--", alpha=0.5, label="Moderate threshold")
    axes[0].axvline(x=30, color="red", linestyle="--", alpha=0.5, label="Severe threshold")
    axes[0].legend(fontsize=8)

    # --- Pie chart of categories ---
    cat_counts = {}
    cat_colors = {"on_time": "#2ecc71", "slight": "#f1c40f", "moderate": "#e67e22", "severe": "#e74c3c"}
    for cat in ["on_time", "slight", "moderate", "severe"]:
        count = categories.count(cat)
        if count > 0:
            cat_counts[cat] = count

    labels = list(cat_counts.keys())
    sizes = list(cat_counts.values())
    colors = [cat_colors.get(c, "gray") for c in labels]
    axes[1].pie(sizes, labels=labels, colors=colors, autopct="%1.0f%%", startangle=90)
    axes[1].set_title("Delay Category Breakdown")

    # --- Confidence distribution ---
    axes[2].hist(confidences, bins=15, color="#9b59b6", edgecolor="white", alpha=0.8)
    axes[2].set_xlabel("Confidence score")
    axes[2].set_ylabel("Flight count")
    axes[2].set_title("Confidence Distribution")

    plt.suptitle(f"Delay Predictions ({len(delays)} flights)", fontsize=13)
    plt.tight_layout()
    plt.show()

    # Summary stats
    print(f"\nSummary:")
    print(f"  Flights: {len(delays)}")
    print(f"  Avg delay: {sum(delay_values)/len(delay_values):.1f} min")
    print(f"  Max delay: {max(delay_values):.1f} min")
    print(f"  Avg confidence: {sum(confidences)/len(confidences):.2f}")
    for cat in ["on_time", "slight", "moderate", "severe"]:
        cnt = categories.count(cat)
        pct = cnt / len(delays) * 100
        print(f"  {cat}: {cnt} ({pct:.0f}%)")